In [ ]:
import sys, subprocess
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], capture_output=True, text=True)
    print("OK    " if r.returncode == 0 else "FAILED", pkgs)
    if r.returncode != 0: print(r.stderr[-1500:])

pip("bitsandbytes>=0.46.1", "accelerate", "peft", "datasets", "scikit-learn")
pip("transformers==5.0.0", "huggingface_hub>=1.3,<2")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must be set BEFORE torch is imported
import torch, transformers, huggingface_hub, peft, datasets, bitsandbytes, sklearn
print("transformers", transformers.__version__, "| hub", huggingface_hub.__version__, "| peft", peft.__version__,
      "| datasets", datasets.__version__, "| bitsandbytes", bitsandbytes.__version__, "| sklearn", sklearn.__version__)
assert transformers.__version__.startswith("5."), f"transformers must be 5.x, got {transformers.__version__}"
from transformers.models.mistral.configuration_mistral import MistralConfig
assert torch.cuda.is_available(), "Enable a GPU: Kaggle > Settings > Accelerator"
assert torch.cuda.device_count() == 1, "CUDA_VISIBLE_DEVICES was set too late"
print("ALL GOOD")


In [ ]:
import json, math, re, hashlib, os
import numpy as np

# ---------------------------------------------------------------------------------------------------
# THE FRAGMENT STANDARD. Edit it in ONE place, then paste this identical cell into EVERY notebook
# (tone, json, sql, legal, merge lab). HARD = must match for adapters to be mergeable.
# ---------------------------------------------------------------------------------------------------
FRAGMENT_STANDARD = {
    "schema": "fragment-standard/v1",
    "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    "base_revision": "63a8b081895390a26e140280378bc85ec8bce07a",  # HARD: pinned commit SHA
    "chat_template_sha256": "796853172235ab3aaa4810013ddfe2eab481bda3306dad4f700c00d436eed596",  # HARD: pinned chat template SHA
    "quantization": "bnb-nf4-double-quant, fp16-compute",
    "lora": {                         # HARD: r, alpha, target_modules, use_rslora, use_dora
        "r": 16, "alpha": 32, "dropout": 0.05, "bias": "none", "use_rslora": False, "use_dora": False,
        "target_modules": ["down_proj", "gate_proj", "k_proj", "o_proj", "q_proj", "up_proj", "v_proj"],
    },
    "training": {                     # SOFT: recorded, recommended identical, not merge-breaking
        "seed": 42, "lr": 2e-4, "scheduler": "linear", "warmup_frac": 0.03, "epochs": 2, "optimizer": "adamw_torch",
        "weight_decay": 0.01, "effective_batch": 16, "loss": "completion-only + EOS appended",
        "overlong": "drop, never truncate", "best_model": "load_best_model_at_end on eval_loss", "save_total_limit": 3,
    },
}
STD = FRAGMENT_STANDARD
SL = STD["lora"]
STD_SCALING = SL["alpha"] / SL["r"]
ATTN = {"q_proj", "k_proj", "v_proj", "o_proj"}
KEY_RE = re.compile(r"layers\.(\d+)\.(?:self_attn|mlp)\.(\w+_proj)\.lora_([AB])(?:\.\w+)?\.weight$")

def module_dims(hidden=4096, n_heads=32, n_kv=8, inter=14336):
    """module -> (in_features, out_features) for a Mistral/Llama-style block"""
    kv = hidden // n_heads * n_kv
    return {"q_proj": (hidden, hidden), "k_proj": (hidden, kv), "v_proj": (hidden, kv), "o_proj": (hidden, hidden),
            "gate_proj": (hidden, inter), "up_proj": (hidden, inter), "down_proj": (inter, hidden)}

def peft_key(layer, module, ab):
    blk = "self_attn" if module in ATTN else "mlp"
    return f"base_model.model.model.layers.{layer}.{blk}.{module}.lora_{ab}.weight"

def compat_key(base, base_rev, r, alpha, targets, rslora, dora, tmpl_sha, quant=STD["quantization"]):
    s = json.dumps([base, base_rev, r, alpha, sorted(targets), bool(rslora), bool(dora), tmpl_sha, quant])
    return hashlib.sha256(s.encode()).hexdigest()[:16]

def check_core(cfg, shapes, reg=None, dims=None, n_layers=32, std=STD):
    """Pure function (no GPU / file IO). Returns (rows, verdict). status: OK | ALIGNABLE | FAIL | UNKNOWN"""
    L = std["lora"]; dims = dims or module_dims(); reg = reg or {}; rows = []
    add = lambda name, status, detail="": rows.append(("HARD", name, status, str(detail)))
    ok = lambda c: "OK" if c else "FAIL"
    add("peft_type is LORA", ok(cfg.get("peft_type") == "LORA"), cfg.get("peft_type"))
    add("base model", ok(cfg.get("base_model_name_or_path") == std["base_model"]), cfg.get("base_model_name_or_path"))
    r, alpha, rs = cfg.get("r"), cfg.get("lora_alpha"), bool(cfg.get("use_rslora", False))
    scaling = alpha / (math.sqrt(r) if rs else r)
    add("rank r", "OK" if r == L["r"] else "ALIGNABLE", f"{r} (standard {L['r']})" + (" - larger: lossy SVD truncation" if r > L["r"] else ""))
    add("lora_alpha / scaling", "OK" if (alpha == L["alpha"] and r == L["r"] and not rs) else "ALIGNABLE", f"alpha {alpha}, scaling {scaling:g} (standard {STD_SCALING:g})")
    tm = cfg.get("target_modules")
    if isinstance(tm, (list, set, tuple)):
        tms, std_t = set(tm), set(L["target_modules"])
        add("target_modules", "OK" if tms == std_t else ("FAIL" if tms - std_t else "ALIGNABLE"), sorted(tms))
    else:
        add("target_modules", "FAIL", f"regex/str target_modules not supported: {tm!r}")
    add("use_dora", ok(not cfg.get("use_dora", False)))
    add("use_rslora", "OK" if rs == L["use_rslora"] else "ALIGNABLE", rs)
    add("no rank/alpha patterns", ok(not cfg.get("rank_pattern") and not cfg.get("alpha_pattern")))
    add("no modules_to_save (extra full weights)", ok(not cfg.get("modules_to_save")), cfg.get("modules_to_save"))
    bad = [k for k in shapes if not KEY_RE.search(k)]
    add("only LoRA A/B tensors", ok(not bad), bad[:3])
    shape_bad, present = [], set()
    for k, shp in shapes.items():
        m = KEY_RE.search(k)
        if not m: continue
        layer, mod, ab = int(m.group(1)), m.group(2), m.group(3)
        if mod not in dims: shape_bad.append((k, "unknown module")); continue
        din, dout = dims[mod]
        want = (r, din) if ab == "A" else (dout, r)
        if tuple(shp) != want: shape_bad.append((k, tuple(shp), want))
        present.add((layer, mod, ab))
    add("tensor shapes", ok(not shape_bad), shape_bad[:2])
    full = {(l, m, ab) for l in range(n_layers) for m in L["target_modules"] for ab in "AB"}
    add("coverage (all layers x modules)", "OK" if full <= present else "ALIGNABLE", f"{len(present & full)}/{len(full)} tensors")
    for fld, label in (("base_revision", "base revision"), ("chat_template_sha256", "chat template hash")):
        want, got = std.get(fld), reg.get(fld)
        if want is None or got is None: add(label, "UNKNOWN", f"standard={want} adapter={got}")
        else: add(label, ok(want == got), f"{got}")
    st = [x[2] for x in rows]
    verdict = ("INCOMPATIBLE" if "FAIL" in st else "NEEDS ALIGNMENT" if "ALIGNABLE" in st
               else "COMPATIBLE (some checks unverified)" if "UNKNOWN" in st else "MERGE-READY")
    return rows, verdict

def print_report(name, rows, verdict):
    print(f"\n=== {name} ===")
    for lvl, n, s, d in rows: print(f"  [{s:<9}] {n:<42} {d[:90]}")
    print(f"  -> VERDICT: {verdict}")

def check_adapter_dir(path, dims=None, n_layers=32, std=STD, verbose=True, framework="pt"):
    from safetensors import safe_open
    cfg = json.load(open(f"{path}/adapter_config.json"))
    rp = f"{path}/registry_entry.json"
    reg = json.load(open(rp)) if os.path.exists(rp) else {}
    with safe_open(f"{path}/adapter_model.safetensors", framework=framework) as sf:
        shapes = {k: tuple(sf.get_slice(k).get_shape()) for k in sf.keys()}
    rows, verdict = check_core(cfg, shapes, reg, dims, n_layers, std)
    if verbose: print_report(os.path.basename(path.rstrip("/")), rows, verdict)
    return rows, verdict

def check_set(paths, dims=None, n_layers=32):
    """Every adapter that will be merged together must pass check_adapter_dir AND share one compatibility_key."""
    keys, verdicts = {}, {}
    for p in paths:
        rows, verdicts[p] = check_adapter_dir(p, dims, n_layers)
        rp = f"{p}/registry_entry.json"
        keys[p] = json.load(open(rp)).get("compatibility_key") if os.path.exists(rp) else None
    print("\ncompatibility keys:", keys)
    same = len({k for k in keys.values() if k}) <= 1 and all(keys.values())
    print("ALL SHARE ONE compatibility_key:", "YES" if same else "NO / missing  <- fix before merging")
    return verdicts, same

def lora_delta_norms(path, scaling=STD_SCALING):
    """Frobenius norm of each module's delta  s*B@A  (computed through the r x r Gram matrices; never builds the dense delta)."""
    import torch
    from safetensors.torch import load_file
    t = load_file(f"{path}/adapter_model.safetensors"); pairs = {}
    for k, v in t.items():
        m = KEY_RE.search(k)
        if m: pairs.setdefault((int(m.group(1)), m.group(2)), {})[m.group(3)] = v.float()
    norms = {}
    for (l, mod), ab in pairs.items():
        A, B = ab["A"], ab["B"]
        norms[f"{l}.{mod}"] = float(scaling * torch.sqrt(torch.clamp(torch.trace((B.T @ B) @ (A @ A.T)), min=0)))
    v = np.array(list(norms.values()))
    return {"mean": float(v.mean()), "max": float(v.max()), "l2_total": float(np.sqrt((v ** 2).sum())), "n_modules": len(v)}

# ------------------------------- aligning an EXTERNAL adapter to the standard -------------------------------
def align_module(A, B, r_ext, alpha_ext, rslora, r_std=SL["r"], alpha_std=SL["alpha"]):
    """Re-express  delta = s_ext * B@A  as  s_std * B2@A2  with rank r_std. Exact if r_ext <= r_std, best rank-r_std approx otherwise."""
    s_ext = alpha_ext / (math.sqrt(r_ext) if rslora else r_ext); s_std = alpha_std / r_std
    A = A.astype(np.float32); B = B.astype(np.float32)
    if r_ext <= r_std:
        pad = r_std - r_ext
        A2 = np.vstack([A, np.zeros((pad, A.shape[1]), np.float32)])
        B2 = np.hstack([B * (s_ext / s_std), np.zeros((B.shape[0], pad), np.float32)])
        return A2, B2, 0.0
    Qb, Rb = np.linalg.qr(B)                 # B: out x r_ext
    Qa, Ra = np.linalg.qr(A.T)               # A.T: in x r_ext
    U, S, Vt = np.linalg.svd(s_ext * (Rb @ Ra.T))
    err = float(np.sqrt((S[r_std:] ** 2).sum() / max((S ** 2).sum(), 1e-30)))
    Ur, Sr, Vtr = U[:, :r_std], S[:r_std], Vt[:r_std]
    f = np.sqrt(Sr / s_std)
    return (f[:, None] * (Vtr @ Qa.T)).astype(np.float32), ((Qb @ Ur) * f).astype(np.float32), err

def align_tensors(tensors, cfg, dims=None, n_layers=32, std=STD):
    dims = dims or module_dims(); L = std["lora"]
    r_ext, a_ext, rs = cfg["r"], cfg["lora_alpha"], bool(cfg.get("use_rslora", False))
    pairs = {}
    for k, v in tensors.items():
        m = KEY_RE.search(k)
        if m: pairs.setdefault((int(m.group(1)), m.group(2)), {})[m.group(3)] = np.asarray(v, dtype=np.float32)
    out, errs = {}, []
    for layer in range(n_layers):
        for mod in L["target_modules"]:
            din, dout = dims[mod]; ab = pairs.get((layer, mod))
            if ab and "A" in ab and "B" in ab:
                A2, B2, e = align_module(ab["A"], ab["B"], r_ext, a_ext, rs, L["r"], L["alpha"]); errs.append(e)
            else:                                           # module the external adapter did not train -> explicit zero delta
                A2, B2 = np.zeros((L["r"], din), np.float32), np.zeros((dout, L["r"]), np.float32)
            out[peft_key(layer, mod, "A")], out[peft_key(layer, mod, "B")] = A2, B2
    n_all = n_layers * len(L["target_modules"])
    return out, {"max_rel_error": max(errs) if errs else 0.0, "modules_converted": len(errs), "modules_zero_filled": n_all - len(errs)}

def align_adapter(src_dir, dst_dir, dims=None, n_layers=32):
    """Convert a compatible-base external adapter into the standard format. Refuses INCOMPATIBLE adapters."""
    import torch
    from safetensors.torch import load_file, save_file
    from peft import LoraConfig
    rows, verdict = check_adapter_dir(src_dir, dims, n_layers)
    if verdict == "INCOMPATIBLE":
        raise ValueError("Cannot align: " + "; ".join(f"{n}: {d}" for _, n, s, d in rows if s == "FAIL"))
    cfg = json.load(open(f"{src_dir}/adapter_config.json"))
    t = {k: v.float().numpy() for k, v in load_file(f"{src_dir}/adapter_model.safetensors").items()}
    out, rep = align_tensors(t, cfg, dims, n_layers)
    os.makedirs(dst_dir, exist_ok=True)
    save_file({k: torch.from_numpy(v).contiguous() for k, v in out.items()}, f"{dst_dir}/adapter_model.safetensors")
    LoraConfig(r=SL["r"], lora_alpha=SL["alpha"], lora_dropout=SL["dropout"], bias=SL["bias"], target_modules=SL["target_modules"],
               use_rslora=False, use_dora=False, task_type="CAUSAL_LM", base_model_name_or_path=STD["base_model"]).save_pretrained(dst_dir)
    h = hashlib.sha256(open(f"{src_dir}/adapter_model.safetensors", "rb").read()).hexdigest()
    json.dump({"registry_schema": "fragment-registry/v1", "fragment_id": os.path.basename(src_dir.rstrip("/")) + "-aligned",
               "aligned_from": {"path": src_dir, "r": cfg["r"], "lora_alpha": cfg["lora_alpha"], "target_modules": cfg.get("target_modules"),
                                "use_rslora": cfg.get("use_rslora", False), "source_sha256": h},
               "alignment": rep, "lossless": rep["max_rel_error"] == 0.0,
               "base_model": STD["base_model"], "lora_scaling": STD_SCALING,
               "base_revision_verified": False,   # we cannot verify which exact base commit an external adapter was trained on
               "compatibility_key": (compat_key(STD["base_model"], STD["base_revision"], SL["r"], SL["alpha"], SL["target_modules"], False, False,
                                                STD["chat_template_sha256"]) if STD["base_revision"] and STD["chat_template_sha256"] else None)},
              open(f"{dst_dir}/registry_entry.json", "w"), indent=2)
    print("aligned ->", dst_dir, rep)
    return rep

def stamp_registry(path, base_revision, chat_template_sha256, fragment_id=None):
    """For an adapter that already exists (e.g. the json adapter): add the fields the merge system needs, computed from its OWN adapter_config."""
    cfg = json.load(open(f"{path}/adapter_config.json"))
    rp = f"{path}/registry_entry.json"
    reg = json.load(open(rp)) if os.path.exists(rp) else {}
    tm = cfg.get("target_modules")
    reg.update({"registry_schema": "fragment-registry/v1", "base_model": cfg.get("base_model_name_or_path"),
                "base_revision": base_revision, "chat_template_sha256": chat_template_sha256,
                "lora": {"r": cfg.get("r"), "alpha": cfg.get("lora_alpha"), "target_modules": sorted(tm) if isinstance(tm, (list, set)) else tm,
                         "use_rslora": bool(cfg.get("use_rslora", False)), "use_dora": bool(cfg.get("use_dora", False))},
                "compatibility_key": compat_key(cfg.get("base_model_name_or_path"), base_revision, cfg.get("r"), cfg.get("lora_alpha"),
                                                tm if isinstance(tm, (list, set)) else [], cfg.get("use_rslora", False), cfg.get("use_dora", False), chat_template_sha256)})
    if fragment_id: reg["fragment_id"] = fragment_id
    if os.path.exists(f"{path}/adapter_model.safetensors"):
        reg["adapter_size_bytes"] = os.path.getsize(f"{path}/adapter_model.safetensors")
        reg["sha256"] = hashlib.sha256(open(f"{path}/adapter_model.safetensors", "rb").read()).hexdigest()
    json.dump(reg, open(rp, "w"), indent=2)
    return reg["compatibility_key"]


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json, re, math, time, hashlib, random, datetime, shutil, inspect, contextlib
from collections import Counter, defaultdict
import numpy as np
import torch

assert torch.cuda.is_available(), "Enable a GPU: Kaggle > Settings > Accelerator"
assert torch.cuda.device_count() == 1, "restart the kernel, run Step A/B again first"

SMOKE_TEST   = False         # True = tiny dry run of the WHOLE pipeline. False = real run.

FRAGMENT_ID  = "support-reply-tone"
DATASET      = "Tobi-Bueck/customer-support-tickets"
BASE_MODEL   = STD["base_model"]
SEED         = STD["training"]["seed"]
EPOCHS       = 1 if SMOKE_TEST else 2      # standard 2 epochs (3 epochs overfit after step 400)
BATCH, ACCUM = 2, 8                                   # effective batch 16 = standard
assert BATCH * ACCUM == STD["training"]["effective_batch"]

NUM_TRAIN_EXAMPLES = 24 if SMOKE_TEST else 2800  # 2,800 train + 200 held-out = 3,000 (safe buffer for clean pool)
EVAL_POOL    = 10 if SMOKE_TEST else 200              # held-out clusters: VAL_N for checkpoint selection, rest = untouched test
VAL_N        = 4  if SMOKE_TEST else 50
GEN_N        = 4  if SMOKE_TEST else 150
NEUTRAL_N    = 4  if SMOKE_TEST else 40 # test rows used for generation + judge
AUDIT_N      = 4  if SMOKE_TEST else 30               # pairs exported for your own manual scoring
MAX_LEN      = 1024
MAX_STEPS    = 2  if SMOKE_TEST else -1
NEAR_DUP_THR, MAX_PER_CLUSTER = 0.6, 2     # tighter 0.6 threshold to eliminate train-eval leak
GEN_CFG      = dict(do_sample=False, max_new_tokens=400)   # frozen for every evaluation (re-set from data in Cell 4)
WIN_GATE     = 0.65                                   # adapter must beat base in >= 65% of judged pairs

OUT_ROOT    = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SUFFIX      = "-smoke" if SMOKE_TEST else ""
ADAPTER_DIR = f"{OUT_ROOT}/support-reply-tone-adapter{SUFFIX}"
CKPT_DIR    = f"{OUT_ROOT}/support-reply-tone-ckpt{SUFFIX}"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("config ok | smoke:", SMOKE_TEST)

# ---- prompt contract (recorded in registry for the router) ----
PROMPT_CONTRACT = {
    "format": "bare customer ticket (Subject: ... \n\n body)",
    "system_prompt": None,  # adapter was trained WITHOUT a system prompt
    "output": "warm empathetic plain-text reply",
    "note": "the prompted-base rival uses a system prompt; the adapter encodes it implicitly"
}


In [ ]:
from datasets import load_dataset
raw_ds = load_dataset(DATASET, split="train")
print(raw_ds); print("columns:", raw_ds.column_names)
rows_all = list(raw_ds)
print("languages:", Counter(r.get("language") for r in rows_all).most_common(6))
en_rows = [r for r in rows_all if str(r.get("language", "")).lower().startswith("en")]
print("English rows:", len(en_rows))
try:
    from huggingface_hub import dataset_info
    DATASET_REVISION = dataset_info(DATASET).sha
except Exception as e:
    DATASET_REVISION = None; print("could not read dataset revision:", repr(e))
print("dataset revision:", DATASET_REVISION)
for r in en_rows[:2]:
    print("\n--- SUBJECT ---", r.get("subject")); print("--- BODY ---", (r.get("body") or "")[:250]); print("--- ANSWER ---", (r.get("answer") or "")[:350])

In [ ]:
# ---- one skill: warm, acknowledging plain-text replies. No JSON, no priority field. ----
# Catches both <name>, <customer_name> and [name], [Customer Name], [Agent Name]
PLACEHOLDER   = re.compile(r"(?:<[A-Za-z_ ]{1,30}>|\[[A-Za-z_ ]{1,30}\])")
GREETING_NAME = re.compile(r"^\s*(?:dear|hello|hi)\s*(?:<[A-Za-z_ ]{1,30}>|\[[A-Za-z_ ]{1,30}\])\s*[,:]?\s*", re.I)
EMPATHY = re.compile(r"\b(sorry|apolog\w*|regret\w*|inconvenien\w*|frustrat\w*|understand (?:how|that|the|your|this|why)|"
                     r"i can (?:see|imagine|understand)|appreciate your (?:patience|understanding|feedback)|"
                     r"thank you for (?:bringing|letting|your patience|your understanding)|we (?:truly |really )?value)\b", re.I)

def fix_reply(a):
    a = GREETING_NAME.sub("Hello,\n\n", a.strip())
    a = re.sub(r"^(Hello,\n\n)([a-z])", lambda m: m.group(1) + m.group(2).upper(), a)
    a = re.sub(r"\s*,?\s*(?:<[A-Za-z_ ]{1,30}>|\[[A-Za-z_ ]{1,30}\])(?=\s*[.,!?:;])", "", a)
    lines = a.splitlines()
    while lines and (not lines[-1].strip() or PLACEHOLDER.fullmatch(lines[-1].strip())): lines.pop()
    a = "\n".join(lines).strip()
    return None if PLACEHOLDER.search(a) else a

def clean_row(r):
    """returns (row, reason). reason None = usable tone example; reason set with a row = valid ticket whose reply has NO apology marker."""
    body, subj, ans = r.get("body"), r.get("subject"), r.get("answer")
    if not body or not ans: return None, "missing field"
    ans = fix_reply(ans)
    if ans is None: return None, "placeholder left in reply"
    if not (30 <= len(ans) <= 1200): return None, "reply length"
    body = PLACEHOLDER.sub(lambda m: "[" + m.group(0).strip("<>[]") + "]", body.strip())
    if len(body) < 20: return None, "short body"
    row = {"prompt": f"Subject: {(subj or '(no subject)').strip()}\n\n{body}", "reply": ans, "completion": ans}
    return row, (None if EMPATHY.search(ans) else "no acknowledgement/apology marker")

base_rate = np.mean([bool(EMPATHY.search(r.get("answer") or "")) for r in en_rows])
print(f"share of ALL English replies that already carry an empathy/apology marker: {base_rate:.1%}")
clean, neutral_pool, dropped = [], [], Counter()
for r in en_rows:
    row, why = clean_row(r)
    if row is not None and why is None: clean.append(row)
    else:
        dropped[why] += 1
        if row is not None: neutral_pool.append(row)          # kept aside: ORDINARY tickets for the over-apology check (Cell 9e)
print("usable tone rows:", len(clean), "| dropped:", dict(dropped), "| ordinary-ticket pool:", len(neutral_pool))
# ---- MIX IN NEUTRAL-BUT-PROFESSIONAL REPLIES (~15%) to prevent over-apologising ----
NEUTRAL_MIX_FRAC = 0.15  # fraction of training rows that are neutral (no empathy marker)
n_neutral_mix = int(NUM_TRAIN_EXAMPLES * NEUTRAL_MIX_FRAC)
if neutral_pool and n_neutral_mix > 0:
    random.Random(SEED + 3).shuffle(neutral_pool)
    neutral_for_train = neutral_pool[:n_neutral_mix]
    clean = clean + neutral_for_train
    random.Random(SEED).shuffle(clean)
    print(f"mixed in {len(neutral_for_train)} neutral-professional replies ({NEUTRAL_MIX_FRAC:.0%}) to prevent over-apologising")
else:
    print("no neutral replies mixed in (pool empty or NEUTRAL_MIX_FRAC=0)")
assert len(clean) >= NUM_TRAIN_EXAMPLES + EVAL_POOL, "too few rows survive - loosen EMPATHY or the length limits"

# ---- near-duplicate clusters (TF-IDF cosine >= NEAR_DUP_THR) -> split by CLUSTER, never by row ----
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, ngram_range=(1, 2), max_features=200000)
X = vec.fit_transform([c["prompt"] for c in clean]).tocsr()
n = X.shape[0]; parent = list(range(n))
def find(i):
    while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
    return i
for s in range(0, n, 500):
    S = (X[s:s + 500] @ X.T).tocoo()
    m = S.data >= NEAR_DUP_THR
    for i, j in zip(S.row[m] + s, S.col[m]):
        if j > i:
            a, b = find(i), find(j)
            if a != b: parent[b] = a
clusters = defaultdict(list)
for i in range(n): clusters[find(i)].append(i)
cl = list(clusters.values()); sizes = np.array([len(x) for x in cl])
print(f"{n} rows -> {len(cl)} near-duplicate clusters | largest {sizes.max()} | singletons {int((sizes == 1).sum())}")

rng = random.Random(SEED); rng.shuffle(cl)
eval_idx = [rng.choice(c) for c in cl[:EVAL_POOL]]                          # one row per held-out cluster; its siblings are DISCARDED
pool = [i for c in cl[EVAL_POOL:] for i in rng.sample(c, min(len(c), MAX_PER_CLUSTER))]
rng.shuffle(pool)
if len(pool) < NUM_TRAIN_EXAMPLES:
    print(f"NOTICE: Available pool ({len(pool)}) < requested ({NUM_TRAIN_EXAMPLES}) due to tighter NEAR_DUP_THR=0.6 clustering.")
    print(f"Automatically adjusting training rows to full available pool: {len(pool)}")
train_idx = pool[:min(len(pool), NUM_TRAIN_EXAMPLES)]
print(f"train {len(train_idx)} of {min(len(pool), NUM_TRAIN_EXAMPLES)} requested (pool after per-cluster cap: {len(pool)}) | held-out {len(eval_idx)} (val {VAL_N}, test {len(eval_idx) - VAL_N})")

# ---- INDEPENDENT leak check: the clustering used NEAR_DUP_THR, so testing against it proves nothing. Look at looser similarity and READ the pairs. ----
Sd = (X[eval_idx] @ X[train_idx].T).tocsr()
sim_max = np.asarray(Sd.max(axis=1).todense()).ravel(); best_j = np.asarray(Sd.argmax(axis=1)).ravel()
print("nearest-train-neighbour cosine of each held-out ticket: " + " | ".join(f"share >= {t}: {(sim_max >= t).mean():.0%}" for t in (0.5, 0.6, 0.7)) + f" | max {sim_max.max():.2f}")
for k in np.argsort(-sim_max)[:5]:
    print(f"\n  cos {sim_max[k]:.2f}\n    HELD-OUT: {clean[eval_idx[k]]['prompt'][:170]!r}\n    TRAIN   : {clean[train_idx[best_j[k]]]['prompt'][:170]!r}")
if (sim_max >= 0.6).mean() > 0.10:
    print("\n>>> WARNING: >10% of held-out tickets have a train neighbour at cosine >= 0.6. Set NEAR_DUP_THR = 0.6 in Cell 3 and re-run Cells 1-3 before training.")
train_rows = [clean[i] for i in train_idx]
val_rows   = [clean[i] for i in eval_idx[:VAL_N]]
test_rows  = [clean[i] for i in eval_idx[VAL_N:]]

# ---- ordinary tickets (original agent reply had no apology) that are not near-copies of training tickets ----
if neutral_pool:
    nn = np.asarray((vec.transform([c["prompt"] for c in neutral_pool]) @ X[train_idx].T).max(axis=1).todense()).ravel()
    cand = [c for c, v in zip(neutral_pool, nn) if v < 0.6]
    random.Random(SEED + 1).shuffle(cand); neutral_rows = cand[:NEUTRAL_N]
else:
    neutral_rows = []
print(f"\nordinary-ticket probe set: {len(neutral_rows)} tickets")

# ---- output diversity: formulaic replies make tone trivially detectable ----
openings = Counter(" ".join(r["reply"].split()[:6]).lower() for r in train_rows)
print("top-5 reply openings in TRAIN:", openings.most_common(5), "| distinct openings:", len(openings), "of", len(train_rows))
print("reply chars (train) mean %.0f | p95 %.0f" % (np.mean([len(r["reply"]) for r in train_rows]), np.percentile([len(r["reply"]) for r in train_rows], 95)))
print("example reply:\n", train_rows[0]["reply"][:400])


In [ ]:
from transformers import AutoTokenizer
try:
    from huggingface_hub import model_info
    BASE_REVISION = STD["base_revision"] or model_info(BASE_MODEL).sha
except Exception as e:
    BASE_REVISION = STD["base_revision"]; print("could not resolve base revision:", repr(e))
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.add_bos_token = False
BOS, EOS, PAD = tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id
assert tokenizer("hi")["input_ids"][0] != BOS, "tokenizer is still auto-adding a BOS"
CHAT_TEMPLATE_SHA = hashlib.sha256((tokenizer.chat_template or "").encode()).hexdigest()
print("BASE_REVISION     =", BASE_REVISION)
print("CHAT_TEMPLATE_SHA =", CHAT_TEMPLATE_SHA)
print(">>> Paste both values into FRAGMENT_STANDARD (base_revision, chat_template_sha256) in EVERY notebook so the pin is enforced.")
if STD["base_revision"]: assert STD["base_revision"] == BASE_REVISION
if STD["chat_template_sha256"]: assert STD["chat_template_sha256"] == CHAT_TEMPLATE_SHA, "chat template differs from the standard"

def render_prompt(text):
    return tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)

boundary_fallbacks = 0
total_encoded = 0

def encode(prompt_text, completion):
    global boundary_fallbacks, total_encoded
    total_encoded += 1
    p = render_prompt(prompt_text)
    p_ids = tokenizer(p, add_special_tokens=False)["input_ids"]
    full = tokenizer(p + completion, add_special_tokens=False)["input_ids"]
    if full[:len(p_ids)] != p_ids:
        boundary_fallbacks += 1
        full = p_ids + tokenizer(completion, add_special_tokens=False)["input_ids"]
    full = full + [EOS]
    return {"input_ids": full, "labels": [-100] * len(p_ids) + full[len(p_ids):]}

def prep(rs, name):
    out, too_long = [], 0
    for r in rs:
        e = encode(r["prompt"], r["completion"])
        if len(e["input_ids"]) > MAX_LEN: too_long += 1; continue
        assert e["input_ids"].count(BOS) == 1 and e["labels"][-1] == EOS and any(l != -100 for l in e["labels"])
        out.append({**r, "enc": e})
    print(f"{name}: kept {len(out)}, dropped {too_long} longer than {MAX_LEN} tokens (dropped, NOT truncated)")
    return out
train_rows, val_rows, test_rows = prep(train_rows, "train"), prep(val_rows, "val"), prep(test_rows, "test")
assert train_rows and val_rows and test_rows
print(f"tokenization boundary fallbacks: {boundary_fallbacks}/{max(1, total_encoded)} ({boundary_fallbacks/max(1, total_encoded):.1%})")

C_ = np.array([sum(l != -100 for l in r["enc"]["labels"]) for r in train_rows])
L_ = np.array([len(r["enc"]["input_ids"]) for r in train_rows])
print(f"total tokens mean {L_.mean():.0f} | p95 {np.percentile(L_, 95):.0f} | max {L_.max()}   completion tokens mean {C_.mean():.0f} | p95 {np.percentile(C_, 95):.0f}")
print("frozen generation settings (same for adapter, base and prompted base):", GEN_CFG, "| train reply p95 tokens:", int(np.percentile(C_, 95)))
e = train_rows[0]["enc"]
print("decoded start:", repr(tokenizer.decode(e["input_ids"][:12])))
print("decoded completion:", repr(tokenizer.decode([t for t in e["labels"] if t != -100])[:200]))


In [ ]:
from transformers import AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION, quantization_config=bnb_config,
                                             device_map={"": 0}, dtype=torch.float16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
lora_config = LoraConfig(r=SL["r"], lora_alpha=SL["alpha"], lora_dropout=SL["dropout"], bias=SL["bias"],
                         target_modules=SL["target_modules"], use_rslora=SL["use_rslora"], use_dora=SL["use_dora"], task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()
model.print_trainable_parameters()

mc = AutoConfig.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
DIMS, N_LAYERS = module_dims(mc.hidden_size, mc.num_attention_heads, mc.num_key_value_heads, mc.intermediate_size), mc.num_hidden_layers
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
expected = N_LAYERS * sum(SL["r"] * (i + o) for i, o in DIMS.values())
assert n_train == expected, f"trainable params {n_train} != expected {expected} for the standard config"
print("trainable params match the standard:", f"{n_train:,}")

In [ ]:
import importlib.metadata as md
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

REGISTRY = {}
def write_registry(**extra):
    REGISTRY.update({
        "registry_schema": "fragment-registry/v1", "fragment_id": FRAGMENT_ID, "version": "1.0.0",
        "fragment_type": "style - always-on, plain-text output (task fragments such as sql/legal/json supply the content)",
        "task": "write a warm, acknowledging customer-support reply",
        "prompt_contract": {"input": "Subject: {subject}\\n\\n{body}  (customer message only, no instruction scaffolding)",
                            "output": "plain-text reply, no JSON", "chat_template": "tokenizer.apply_chat_template, add_generation_prompt=True"},
        "base_model": BASE_MODEL, "base_revision": BASE_REVISION, "chat_template_sha256": CHAT_TEMPLATE_SHA,
        "quantization": STD["quantization"], "standard_schema": STD["schema"],
        "lora": {**SL, "target_modules": sorted(SL["target_modules"])}, "lora_scaling": STD_SCALING,
        "compatibility_key": compat_key(BASE_MODEL, BASE_REVISION, SL["r"], SL["alpha"], SL["target_modules"], SL["use_rslora"], SL["use_dora"], CHAT_TEMPLATE_SHA),
        "training": {**STD["training"], "epochs_run": EPOCHS, "num_train_examples": len(train_rows), "max_len": MAX_LEN, "smoke_test": SMOKE_TEST},
        "dataset": {"name": DATASET, "revision": DATASET_REVISION, "license": "CC-BY-NC-4.0 (non-commercial)", "nature": "synthetic support tickets",
                    "language": "English only", "filters": "replies with acknowledgement/apology marker; placeholders removed; near-duplicate clusters split",
                    "empathy_regex": EMPATHY.pattern, "near_dup_threshold": NEAR_DUP_THR},
        "references": ["Hu et al. 2021 (LoRA)", "Rashkin et al. ACL 2019 (inspiration for empathetic replies only; not trained on)"],
        "adapter_file": "adapter_model.safetensors",
        "adapter_size_bytes": os.path.getsize(f"{ADAPTER_DIR}/adapter_model.safetensors"),
        "sha256": sha256_file(f"{ADAPTER_DIR}/adapter_model.safetensors"),
        "env": {k: md.version(k) for k in ("transformers", "peft", "torch", "bitsandbytes", "datasets")},
        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    })
    REGISTRY.update(extra)
    with open(f"{ADAPTER_DIR}/registry_entry.json", "w") as f:
        json.dump(REGISTRY, f, indent=2, default=lambda o: sorted(o) if isinstance(o, set) else str(o))
print("helpers ok")

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

class DS(torch.utils.data.Dataset):
    def __init__(self, rs): self.x = [r["enc"] for r in rs]
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]

def collate(feats):
    n = max(len(f["input_ids"]) for f in feats)
    ids = torch.full((len(feats), n), PAD, dtype=torch.long)
    lab = torch.full((len(feats), n), -100, dtype=torch.long)
    att = torch.zeros((len(feats), n), dtype=torch.long)
    for i, f in enumerate(feats):
        k = len(f["input_ids"])
        ids[i, :k] = torch.tensor(f["input_ids"]); lab[i, :k] = torch.tensor(f["labels"]); att[i, :k] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

steps_per_epoch = math.ceil(len(train_rows) / (BATCH * ACCUM))
total_steps = MAX_STEPS if MAX_STEPS > 0 else steps_per_epoch * EPOCHS
EVAL_EVERY = 1 if SMOKE_TEST else 50          # eval_steps must equal save_steps for load_best_model_at_end
T = STD["training"]
desired_args = dict(
    output_dir=CKPT_DIR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
    num_train_epochs=EPOCHS, max_steps=MAX_STEPS, learning_rate=T["lr"], lr_scheduler_type=T["scheduler"],
    warmup_steps=max(1, int(T["warmup_frac"] * total_steps)), weight_decay=T["weight_decay"], optim=T["optimizer"],
    fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10, eval_strategy="steps", eval_steps=EVAL_EVERY, save_strategy="steps", save_steps=EVAL_EVERY,
    save_total_limit=T["save_total_limit"], load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    remove_unused_columns=False, report_to="none", seed=SEED, dataloader_num_workers=0)
accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" not in accepted and "evaluation_strategy" in accepted:
    desired_args["evaluation_strategy"] = desired_args.pop("eval_strategy")
rejected = [k for k in desired_args if k not in accepted]
if rejected: print("TrainingArguments rejected on this install (dropped):", rejected)
assert "load_best_model_at_end" in accepted, "Transformers version must support load_best_model_at_end"
args = TrainingArguments(**{k: v for k, v in desired_args.items() if k in accepted})

for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()
trainer = Trainer(model=model, args=args, train_dataset=DS(train_rows), eval_dataset=DS(val_rows), data_collator=collate)
print(f"steps/epoch {steps_per_epoch} | total steps {total_steps} | warmup {desired_args['warmup_steps']}")

base_val = trainer.evaluate()
print("BASE (untrained) validation loss:", round(base_val["eval_loss"], 4))

last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
print("resuming from:", last)
out = trainer.train(resume_from_checkpoint=last)

if SMOKE_TEST:
    full_steps = math.ceil(len(train_rows) / (BATCH * ACCUM)) * T["epochs"]
    sec = out.metrics["train_runtime"] / max(1, MAX_STEPS)
    print(f"\nSMOKE TIMING (upper bound, includes eval): {sec:.1f}s/step -> full run ({full_steps} steps) ~ {sec * full_steps / 3600:.1f} h")

best_val = trainer.evaluate()
print("BEST-checkpoint validation loss:", round(best_val["eval_loss"], 4), "| best ckpt:", trainer.state.best_model_checkpoint)
model.save_pretrained(ADAPTER_DIR); tokenizer.save_pretrained(ADAPTER_DIR)
write_registry(base_val_loss=base_val["eval_loss"], best_val_loss=best_val["eval_loss"], best_checkpoint=str(trainer.state.best_model_checkpoint))
print("adapter + registry saved:", os.listdir(ADAPTER_DIR))


In [ ]:
model.gradient_checkpointing_disable(); model.config.use_cache = True; model.eval()
lora_test = trainer.evaluate(eval_dataset=DS(test_rows), metric_key_prefix="test")["test_loss"]
with model.disable_adapter():
    base_test = trainer.evaluate(eval_dataset=DS(test_rows), metric_key_prefix="basetest")["basetest_loss"]
print(f"held-out TEST loss  base {base_test:.4f}  ->  adapter {lora_test:.4f}   (lower is better; test rows never used for selection)")
write_registry(base_test_loss=base_test, lora_test_loss=lora_test, n_test=len(test_rows))

In [ ]:
# ---------- 9a. frozen generation. THREE conditions, identical greedy settings, truncation MEASURED ----------
SYS = ("You are a customer support agent. Write a warm, empathetic reply that acknowledges the "
       "customer's situation and helps with their issue. Reply with the message text only.\n\n")

@torch.no_grad()
def generate_replies(prompts, use_adapter=True, bs=8):
    model.eval(); outs, cut = [], []; tokenizer.padding_side = "left"
    for i in range(0, len(prompts), bs):
        inp = tokenizer([render_prompt(p) for p in prompts[i:i + bs]], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with (contextlib.nullcontext() if use_adapter else model.disable_adapter()):
            gen = model.generate(**inp, pad_token_id=EOS, use_cache=True, **GEN_CFG)
        for g in gen:
            new = g[inp["input_ids"].shape[1]:]
            cut.append(not bool((new == EOS).any()))            # no EOS emitted => stopped only because max_new_tokens was hit
            outs.append(tokenizer.decode(new, skip_special_tokens=True).strip())
    tokenizer.padding_side = "right"
    return outs, cut

sub = test_rows[:GEN_N]; msgs = [r["prompt"] for r in sub]
t0 = time.time()
# Length-matched prompted baseline system prompt (forces base to be concise like human agents):
SYS_CONCISE = ("You are a customer support agent. Write a concise, warm, empathetic reply in 2-3 sentences (under 100 words) "
               "that acknowledges the customer's situation and helps with their issue. Reply with the message text only.\n\n")

rep_lora, cut_lora   = generate_replies(msgs, True)                                   # adapter, bare ticket (its training format)
rep_base, cut_base   = generate_replies(msgs, False)                                  # base, bare ticket
rep_basep, cut_basep = generate_replies([SYS + m for m in msgs], False)             # base + standard verbose prompt
rep_basep_c, cut_basep_c = generate_replies([SYS_CONCISE + m for m in msgs], False)   # base + length-matched concise prompt
dt = time.time() - t0
print(f"generated 4 x {len(msgs)} replies in {dt:.0f}s  (~{dt / len(msgs) * 150 / 60:.0f} min for 150 rows)")
print(f"truncated at max_new_tokens={GEN_CFG['max_new_tokens']}:  adapter {np.mean(cut_lora):.0%} | base {np.mean(cut_base):.0%} | prompted base {np.mean(cut_basep):.0%}")

# ---- LENGTH ANALYSIS: check whether the judge is biased by reply length ----
len_a, len_b, len_bp = [len(x) for x in rep_lora], [len(x) for x in rep_base], [len(x) for x in rep_basep]
print(f"mean reply length (chars): adapter {np.mean(len_a):.0f} | base {np.mean(len_b):.0f} | prompted base {np.mean(len_bp):.0f}")
print(f"median reply length:       adapter {np.median(len_a):.0f} | base {np.median(len_b):.0f} | prompted base {np.median(len_bp):.0f}")
len_ratio = np.mean(len_a) / max(1, np.mean(len_bp))
if len_ratio < 0.7 or len_ratio > 1.5:
    print(f"WARNING: adapter replies are {'shorter' if len_ratio < 1 else 'longer'} than prompted base by {abs(1 - len_ratio):.0%}. "
          "This may bias the judge. Check if length correlates with judge preference.")

# ---------- 9b. pairwise judge (rubric below), both answer orders, base model with adapter disabled ----------
def wilson(p, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d

def win_stats(votes):
    """votes: 2 = our reply preferred in BOTH answer orders, 1 = order-sensitive split, 0 = other reply preferred in both"""
    n = len(votes)
    if n == 0: return dict(n=0, win=0, split=0, loss=0, win_rate=float("nan"), ci95=[float("nan")] * 2)
    W, tie, Lo = votes.count(2), votes.count(1), votes.count(0)
    wr = (W + 0.5 * tie) / n
    return dict(n=n, win=W, split=tie, loss=Lo, win_rate=wr, ci95=list(wilson(wr, n)))

def fmt(s):
    return f"win {s['win']} | split {s['split']} | loss {s['loss']} of {s['n']}  ->  win rate {s['win_rate']:.1%} (95% CI {s['ci95'][0]:.1%} - {s['ci95'][1]:.1%})"

def cohen_kappa(pairs):
    if not pairs: return float("nan")
    n = len(pairs); cats = sorted({c for p in pairs for c in p})
    po = sum(a == b for a, b in pairs) / n
    pe = sum((sum(a == c for a, _ in pairs) / n) * (sum(b == c for _, b in pairs) / n) for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

JUDGE_TMPL = ("You are evaluating two replies to a customer support message.\n\nCustomer message:\n{msg}\n\nReply A:\n{a}\n\nReply B:\n{b}\n\n"
              "Rubric: prefer the reply that (1) acknowledges the customer's situation and feelings in a warm, human, empathetic tone "
              "(an apology or acknowledgement where fitting) AND (2) still addresses the customer's actual issue with relevant, concrete help. "
              "A warm reply that ignores the issue loses to one that is warm and helpful. Answer with a single letter: A or B.")
LET = {c: sorted({i for i in (tokenizer.convert_tokens_to_ids(c), tokenizer.convert_tokens_to_ids("\u2581" + c)) if i is not None and i != tokenizer.unk_token_id}) for c in "AB"}
assert LET["A"] and LET["B"], "could not find letter tokens"

@torch.no_grad()
def judge_batch(prompts, bs=4):
    picks = []
    for i in range(0, len(prompts), bs):
        chunk = [tokenizer(render_prompt(p), add_special_tokens=False)["input_ids"] for p in prompts[i:i + bs]]
        n_ = max(map(len, chunk))
        ids = torch.full((len(chunk), n_), PAD, dtype=torch.long); att = torch.zeros((len(chunk), n_), dtype=torch.long)
        for j, x in enumerate(chunk): ids[j, :len(x)] = torch.tensor(x); att[j, :len(x)] = 1
        with model.disable_adapter():
            lg = model(input_ids=ids.to(model.device), attention_mask=att.to(model.device)).logits
        for j, x in enumerate(chunk):
            row = lg[j, len(x) - 1]
            picks.append("A" if row[LET["A"]].max() >= row[LET["B"]].max() else "B")
    return picks

def judge_pairs(ms, ours, theirs):
    p1 = [JUDGE_TMPL.format(msg=m, a=a, b=b) for m, a, b in zip(ms, ours, theirs)]     # ours = A
    p2 = [JUDGE_TMPL.format(msg=m, a=b, b=a) for m, a, b in zip(ms, ours, theirs)]     # ours = B (order swapped)
    return [(x == "A") + (y == "B") for x, y in zip(judge_batch(p1), judge_batch(p2))]

votes_p   = judge_pairs(msgs, rep_lora, rep_basep)        # PRIMARY: adapter vs verbose prompted base
votes_pc  = judge_pairs(msgs, rep_lora, rep_basep_c)      # LENGTH-MATCHED: adapter vs concise prompted base
votes_b   = judge_pairs(msgs, rep_lora, rep_base)         # secondary: adapter vs bare base
S_pc      = win_stats(votes_pc)
S_p, S_b = win_stats(votes_p), win_stats(votes_b)
clean_mask = [not (a or b) for a, b in zip(cut_lora, cut_basep)]
S_p_nt = win_stats([v for v, ok in zip(votes_p, clean_mask) if ok])
longer = [len(a) > len(b) for a, b in zip(rep_lora, rep_basep)]
S_long, S_short = win_stats([v for v, l in zip(votes_p, longer) if l]), win_stats([v for v, l in zip(votes_p, longer) if not l])
gate_pass = bool(S_p["win_rate"] >= WIN_GATE and S_p["ci95"][0] > 0.5 and S_pc["ci95"][0] > 0.5)
print("\nPRIMARY        adapter vs VERBOSE prompted base :", fmt(S_p))
print("LENGTH-MATCHED adapter vs CONCISE prompted base :", fmt(S_pc))
print("          same, only pairs neither truncated:", fmt(S_p_nt))
print("          adapter reply longer than rival   :", fmt(S_long))
print("          adapter reply shorter or equal    :", fmt(S_short))
print("secondary adapter vs BARE base      :", fmt(S_b))
print(f"GATE (win rate >= {WIN_GATE:.0%} AND 95% CI lower bound > 50% vs BOTH verbose & concise prompted base): {'PASS' if gate_pass else 'FAIL'}")
if not gate_pass: print("  -> prompting alone matches the adapter (or the judge cannot tell): the fragment is not shown to add tone. Say so in the report; do not rely on it for merging.")
if abs(np.mean(cut_lora) - np.mean(cut_basep)) > 0.10: print("  !! truncation rates differ by >10 points - read the 'neither truncated' line, not the headline")
mk = lambda t: float(np.mean([bool(EMPATHY.search(x)) for x in t]))
print(f"\ntone-marker rate  adapter {mk(rep_lora):.0%} | base {mk(rep_base):.0%} | prompted base {mk(rep_basep):.0%}   (sanity only: the same word list selected the training data)")
print(f"mean reply chars  adapter {np.mean([len(x) for x in rep_lora]):.0f} | base {np.mean([len(x) for x in rep_base]):.0f} | verbose prompted {np.mean([len(x) for x in rep_basep]):.0f} | concise prompted {np.mean([len(x) for x in rep_basep_c]):.0f}")
op_a = Counter(" ".join(x.split()[:6]).lower() for x in rep_lora)
print("top-5 openings, ADAPTER outputs:", op_a.most_common(5), "| distinct:", len(op_a), "of", len(rep_lora))
for i in range(min(2, len(msgs))): print(f"\n--- msg ---\n{msgs[i][:250]}\n--- ADAPTER ---\n{rep_lora[i][:350]}\n--- PROMPTED BASE ---\n{rep_basep[i][:350]}")

# ---------- 9c. blinded pairs (adapter vs PROMPTED base) for YOUR scoring ----------
arng = random.Random(SEED); audit, key = [], []
for i in arng.sample(range(len(msgs)), min(AUDIT_N, len(msgs))):
    flip = arng.random() < 0.5
    X_, Y_ = (rep_basep[i], rep_lora[i]) if flip else (rep_lora[i], rep_basep[i])
    key.append({"id": i, "adapter_is": "Y" if flip else "X", "judge": {2: "adapter", 1: "tie", 0: "base"}[votes_p[i]]})
    audit.append({"id": i, "customer_message": msgs[i], "reply_X": X_, "reply_Y": Y_})
json.dump(audit, open(f"{OUT_ROOT}/judge_audit{SUFFIX}.json", "w"), indent=2)
json.dump(key, open(f"{OUT_ROOT}/judge_audit_key{SUFFIX}.json", "w"), indent=2)
print(f"\nwrote judge_audit{SUFFIX}.json ({len(audit)} blinded pairs). Score them in a RANDOM order if you can, then run Cell 9d.")

def audit_agreement(human):
    """human: ['X','Y','tie',...] in the order of judge_audit.json (the reply YOU prefer)."""
    assert len(human) == len(key), f"need {len(key)} labels"
    hp = ["tie" if h == "tie" else ("adapter" if h == k["adapter_is"] else "base") for h, k in zip(human, key)]
    jp = [k["judge"] for k in key]
    both = [(a, b) for a, b in zip(hp, jp) if a != "tie" and b != "tie"]      # judge-tie (order-sensitive) != human-tie (equal quality)
    print(f"3-way agreement {np.mean([a == b for a, b in zip(hp, jp)]):.0%} | kappa {cohen_kappa(list(zip(hp, jp))):.2f}")
    print(f"non-tie pairs only: n={len(both)} | agreement {np.mean([a == b for a, b in both]) if both else float('nan'):.0%} | kappa {cohen_kappa(both):.2f}")
    xs = sum(h == "X" for h in human); ax = sum(k["adapter_is"] == "X" for k in key)
    on_x = [h == "X" for h, k in zip(human, key) if k["adapter_is"] == "X" and h != "tie"]
    on_y = [h == "X" for h, k in zip(human, key) if k["adapter_is"] == "Y" and h != "tie"]
    print(f"position check: you picked X {xs}/{len(human)}; adapter was X in {ax}/{len(key)}")
    print(f"  P(pick X | adapter was X) = {np.mean(on_x) if on_x else float('nan'):.0%}  vs  P(pick X | adapter was Y) = {np.mean(on_y) if on_y else float('nan'):.0%}")
    print("  (if these two are close your picks do not follow the adapter: position habit or no real difference - re-score ten blind)")
    hw = sum(1 for a in hp if a == "adapter"); ht = sum(1 for a in hp if a == "tie")
    print(f"YOUR win rate for the adapter vs prompted base: {(hw + 0.5 * ht) / len(hp):.0%} (n={len(hp)}; 95% CI {wilson((hw + 0.5 * ht) / len(hp), len(hp))[0]:.0%} - {wilson((hw + 0.5 * ht) / len(hp), len(hp))[1]:.0%})")

write_registry(eval_scores={"judge": {"primary_vs_prompted_base": S_p, "vs_concise_prompted_base": S_pc, "primary_non_truncated": S_p_nt, "vs_bare_base": S_b,
                                       "adapter_longer": S_long, "adapter_not_longer": S_short,
                                       "gate": {"win_rate": WIN_GATE, "ci_lower_above": 0.5, "pass": gate_pass},
                                       "judge_model": "base model, adapter disabled, both orders", "rubric": JUDGE_TMPL, "baseline_system_prompt": SYS},
                            "truncation": {"adapter": float(np.mean(cut_lora)), "base": float(np.mean(cut_base)), "prompted_base": float(np.mean(cut_basep)),
                                           "max_new_tokens": GEN_CFG["max_new_tokens"]},
                            "tone_marker_rate": {"adapter": mk(rep_lora), "base": mk(rep_base), "prompted_base": mk(rep_basep)},
                            "mean_reply_chars": {"adapter": float(np.mean([len(x) for x in rep_lora])), "prompted_base": float(np.mean([len(x) for x in rep_basep]))}})


In [ ]:
# ---------- POSITION-BLINDED HUMAN AUDIT ----------
# Uses judge_audit.json and judge_audit_key.json exported by Cell 11.
# Score each pair (X or Y for the reply you prefer) without knowing which is the adapter.

audit_path = f"{OUT_ROOT}/judge_audit{SUFFIX}.json"
key_path   = f"{OUT_ROOT}/judge_audit_key{SUFFIX}.json"

if os.path.exists(audit_path) and os.path.exists(key_path):
    audit_data = json.load(open(audit_path))
    key_data   = json.load(open(key_path))
    print(f"Loaded {len(audit_data)} blinded pairs from {audit_path}.\n")
    for idx, p in enumerate(audit_data):
        print(f"--- Pair {idx+1}/{len(audit_data)} ---")
        print(f"  Customer Message: {p['customer_message'][:160]}")
        print(f"  Reply X: {p['reply_X'][:700]}")
        print(f"  Reply Y: {p['reply_Y'][:700]}\n")
else:
    print(f"Audit files not found at {audit_path}. Run Cell 11 first.")

# PASTE YOUR BLIND LABELS HERE (e.g. "X" or "Y" or "tie" for each pair):
blind_labels = [
    # 1-10:
    "Y", "X", "Y", "Y", "Y", "X", "X", "Y", "X", "X",
    # 11-20:
    "X", "X", "X", "X", "X", "X", "X", "Y", "Y", "Y",
    # 21-30:
    "Y", "Y", "Y", "Y", "Y", "X", "Y", "Y", "Y", "Y"
]

if blind_labels:
    # Call audit_agreement() directly on judge_audit.json / key from Cell 11
    audit_agreement(blind_labels)
else:
    print("Fill in blind_labels above (one per pair) and re-run this cell to compute human-judge agreement.")


In [ ]:
# ---------- 9e. CONTROL PROBES (Task Arithmetic's "control task" idea): a tone fragment must not bleed into other skills ----------
NEUTRAL_TASKS = [
    # --- original 8 ---
    "Write a SQL query that returns total sales per month from a table orders(order_date, amount).",
    "Convert this list to JSON: apples 3, pears 5, plums 2.",
    "Summarize in one sentence: The committee met on Tuesday and approved the new budget after a short debate.",
    "What is the capital of Australia?",
    "Explain what a primary key is in a database.",
    "Translate to French: The meeting starts at nine.",
    "List three differences between TCP and UDP.",
    "Write a Python function that reverses a string.",
    # --- added: edge cases that resemble but are not support ---
    "Write a formal complaint letter about a delayed shipment.",
    "Describe the plot of a movie where the main character loses everything.",
    "Draft an email declining a job offer politely.",
    "Write a paragraph about the challenges of remote work.",
    "Explain why the Titanic sank.",
    "Write a product description for wireless earbuds.",
    "Create a recipe for chocolate cake with step-by-step instructions.",
    "What are the symptoms of burnout and how to prevent it?",
    "Write a resignation letter that is professional but firm.",
    "Explain the difference between empathy and sympathy.",
    "Draft a message telling a friend you cannot attend their wedding.",
    "Write a review of a restaurant where the food was great but service was slow.",
]
nm = lambda t: float(np.mean([bool(EMPATHY.search(x)) for x in t]))
tk_lora, _ = generate_replies(NEUTRAL_TASKS, True); tk_base, _ = generate_replies(NEUTRAL_TASKS, False)
print(f"NON-SUPPORT tasks (n={len(NEUTRAL_TASKS)}): apology/empathy marker rate  adapter {nm(tk_lora):.0%} vs base {nm(tk_base):.0%}   <- should be ~equal; a gap = tone bleeds into other skills")
for t_, a_ in list(zip(NEUTRAL_TASKS, tk_lora))[:3]: print(f"  task: {t_[:70]}\n  adapter: {a_[:200]!r}")

ord_scores = None
if neutral_rows:
    nmsgs = [r["prompt"] for r in neutral_rows]
    n_lora, _ = generate_replies(nmsgs, True); n_basep, _ = generate_replies([SYS + m for m in nmsgs], False)
    ord_scores = {"adapter": nm(n_lora), "prompted_base": nm(n_basep), "original_human": nm([r["reply"] for r in neutral_rows])}
    print(f"ORDINARY tickets (n={len(nmsgs)}, original agent reply had NO apology): marker rate  adapter {ord_scores['adapter']:.0%} | prompted base {ord_scores['prompted_base']:.0%} | original replies {ord_scores['original_human']:.0%}")
    print("  (adapter far above prompted base here = it over-apologises on neutral requests)")

def composite(fname, message):
    fn = {"name": fname, "description": "Look up an order", "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}
    return ("You have access to the following functions:\n" + json.dumps(fn, indent=2) + f"\n\nConversation:\nUser: {message}\n\n"
            f"Write the JSON arguments for the call to `{fname}`, then on a new line write a short reply to the customer.")

# 10 composite probes across various support requests:
PROBES = [
    composite("get_order_status", "My order 5521 has not arrived after 5 days and I am furious."),
    composite("get_order_status", "Order 8810 arrived damaged. Please tell me where it is in the refund process."),
    composite("get_order_status", "Hi, can you check order 3304 for me? Thanks!"),
    composite("get_order_status", "Where is package 9921? It was supposed to be delivered yesterday."),
    composite("get_order_status", "Tracking says order 1042 was delivered, but I never received it."),
    composite("get_order_status", "I received the wrong item for order 7731. Please assist immediately."),
    composite("get_order_status", "Order 4452 status check please, the tracking link gives an error."),
    composite("get_order_status", "Can you update me on order 6119? It's a birthday gift for tomorrow."),
    composite("get_order_status", "I was charged twice for order 8203. Need this resolved ASAP."),
    composite("get_order_status", "Order 1590 has been stuck in transit for two weeks.")
]

pr_lora, _ = generate_replies(PROBES, True); pr_base, _ = generate_replies(PROBES, False)

def has_json(t):
    start = t.find('{')
    if start == -1: return False
    depth = 0
    for idx in range(start, len(t)):
        if t[idx] == '{': depth += 1
        elif t[idx] == '}':
            depth -= 1
            if depth == 0:
                try: json.loads(t[start:idx+1]); return True
                except Exception: pass
    return False

lora_json_valid = sum(map(has_json, pr_lora))
base_json_valid = sum(map(has_json, pr_base))
print(f"\nvalid JSON args — adapter: {lora_json_valid} | base: {base_json_valid} of {len(PROBES)}")

print("\nCOMPOSITE-FORMAT probes (json-style prompt the tone adapter never saw) - sample:")
for p_, a_, b_ in zip(PROBES[:3], pr_lora[:3], pr_base[:3]):
    print(f"\n  user: {p_.split('User: ')[1][:90]}\n  ADAPTER: {a_[:250]!r}\n  BASE   : {b_[:250]!r}")

write_registry(control_probes={"non_support_marker_rate": {"adapter": nm(tk_lora), "base": nm(tk_base), "n": len(NEUTRAL_TASKS)},
                               "ordinary_tickets": ord_scores,
                               "composite_valid_json": {"adapter": lora_json_valid, "base": base_json_valid, "total": len(PROBES)}})


In [ ]:
# ---------- freeze the evaluation so the MERGED system can be scored on exactly the same things ----------
frozen = {"fragment_id": FRAGMENT_ID, "seed": SEED, "generation": GEN_CFG, "judge_template": JUDGE_TMPL, "win_gate": WIN_GATE,
          "heldout": [{"id": i, "prompt": r["prompt"], "reference_reply": r["reply"]} for i, r in enumerate(test_rows)],
          "gen_subset_ids": list(range(min(GEN_N, len(test_rows)))),"baseline_system_prompt": SYS, "neutral_task_prompts": NEUTRAL_TASKS, "prompt_contract": PROMPT_CONTRACT}
with open(f"{ADAPTER_DIR}/eval_frozen.json", "w") as f: json.dump(frozen, f, indent=2)

# ---------- merge-readiness: standard check, delta norms, hash ----------
rows, verdict = check_adapter_dir(ADAPTER_DIR, DIMS, N_LAYERS)
assert verdict in ("MERGE-READY", "COMPATIBLE (some checks unverified)"), verdict
from safetensors.torch import load_file
w = load_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
assert len(w) == N_LAYERS * len(SL["target_modules"]) * 2, f"expected {N_LAYERS * len(SL['target_modules']) * 2} tensors, got {len(w)}"
assert all(torch.isfinite(t).all() for t in w.values()), "NaN/Inf in adapter"
assert any((t != 0).any() for k, t in w.items() if "lora_B" in k), "all lora_B are zero - nothing was learned"
norms = lora_delta_norms(ADAPTER_DIR)
print("delta norms (scale of this fragment's task vector):", {k: round(v, 3) if isinstance(v, float) else v for k, v in norms.items()})
write_registry(delta_norms=norms, eval_frozen_sha256=sha256_file(f"{ADAPTER_DIR}/eval_frozen.json"), standard_check=verdict)
entry = json.load(open(f"{ADAPTER_DIR}/registry_entry.json"))
assert entry["sha256"] == sha256_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
print("registry OK:", {k: entry[k] for k in ["fragment_id", "compatibility_key", "base_revision", "adapter_size_bytes", "sha256"]})
if not SMOKE_TEST:
    shutil.make_archive(f"{OUT_ROOT}/support-reply-tone-adapter", "zip", ADAPTER_DIR)
    print("zipped ->", f"{OUT_ROOT}/support-reply-tone-adapter.zip")
print("\nSMOKE TEST PASSED - set SMOKE_TEST = False, restart, Step A/B, then Cells 0-10." if SMOKE_TEST else "\nDONE.")